# فصل ۱۳: پردازش داده‌ها در LangChain (Data Processing)
## بخش ۶: پردازش متن‌آگاه (Context-Aware Chunking & Metadata Engineering)

### 🏛️ تراژدی «یتیم شدن چانک‌ها» (Orphaned Chunks)
فرض کنید کتاب راهنمای سفر به ایران را با اسپلیترهای معمولی خرد کرده‌اید:
- **چانک ۵۰:** *«سوغات اصلی آن مسقطی و فالوده است.»*
- **چانک ۵۱:** *«بهترین فصل سفر به آن، اردیبهشت‌ماه است.»*

کاربر در سامانه RAG می‌پرسد: *«سوغات شیراز چیست؟»*
سیستم چانک ۵۰ را به درستی پیدا می‌کند، اما مدل زبانی با خواندن *«سوغات اصلی آن...»* گیج می‌شود: این «آن» به شیراز اشاره دارد؟ بوشهر؟ لار یا جهرم؟ چرا؟
زیرا تیتر **«شهر شیراز»** در چانک شماره ۴۰ جا مانده و چانک‌های بعدی **یتیم و بی‌هویت** شده‌اند!

در این بخش با دو رویکرد مهندسی پیشرفته برای حل این مشکل آشنا می‌شویم:
1. **تقسیم‌بندی ساختاریافته (Structure-Aware Chunking):** استخراج خودکار تیترهای سلسله‌مراتبی مارک‌داون به فراداده‌های هر چانک با `MarkdownHeaderTextSplitter`.
2. **مهندسی متادیتا با هوش مصنوعی (Metadata Tagging):** استفاده از `create_metadata_tagger` و `with_structured_output` برای استخراج خودکار فیلدهای ساخت‌یافته (محصول، احساس، فوریت) و **خلاصه یک‌جمله‌ای برای جستجو (`summary_for_search`)**.

### ۱. شبیه‌سازی مشکل یتیم شدن چانک‌ها با اسپلیترهای عادی

بیایید یک متن گردشگری را با اسپلیتر معمولی خرد کنیم و ببینیم چطور جملات از تیتر شهر جدا شده و مرجع خود را از دست می‌دهند:

In [1]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

travel_guide = """
استان فارس
شهر شیراز
این شهر مهد تمدن و شعر فارسی است.
سوغات اصلی آن مسقطی، کلوچه و فالوده شیرازی است.
بهترین زمان سفر اردیبهشت‌ماه است.
"""

# اسپلیتر ساده با اندازه کوچک
naive_splitter = RecursiveCharacterTextSplitter(chunk_size=55, chunk_overlap=0)
naive_chunks = naive_splitter.split_text(travel_guide.strip())

print(f"تعداد چانک‌های تولیدشده: {len(naive_chunks)}\n")
for i, c in enumerate(naive_chunks, 1):
    print(f"چانک {i}: '{c}'")

تعداد چانک‌های تولیدشده: 3

چانک 1: 'استان فارس
شهر شیراز
این شهر مهد تمدن و شعر فارسی است.'
چانک 2: 'سوغات اصلی آن مسقطی، کلوچه و فالوده شیرازی است.'
چانک 3: 'بهترین زمان سفر اردیبهشت‌ماه است.'


> ⚠️ **مشاهده می‌کنید:** در چانک ۳ متن *«سوغات اصلی آن مسقطی، کلوچه و فالوده...»* بدون هیچ اشاره‌ای به نام «شیراز» ظاهر شده است. اگر این چانک به تنهایی بازیابی شود، مدل مرجع «آن» را نخواهد دانست.

### ۲. راه‌حل اول: تقسیم‌بندی ساختاریافته با `MarkdownHeaderTextSplitter`

در اسناد ساختاریافته (Markdown)، تیترها صرفاً فونت درشت نیستند؛ بلکه **رابطه پدر-فرزندی** را تعریف می‌کنند.
ابزار `MarkdownHeaderTextSplitter` متن را برش می‌زند، اما تیترهای بالاسری را در **کوله‌پشتی (`metadata`)** هر چانک قرار می‌دهد:

In [2]:
from langchain_text_splitters import MarkdownHeaderTextSplitter

markdown_text = """
# راهنمای ایرانگردی
## استان فارس
### شهر شیراز
این شهر مهد تمدن است. سوغات آن مسقطی و فالوده است.
### شهر مرودشت
تخت جمشید در اینجا قرار دارد.
## استان اصفهان
### شهر اصفهان
نصف جهان لقب این شهر است.
"""

# تعریف رابطه سطوح تیتر با کلیدهای متادیتا
headers_to_split_on = [
    ("#", "category"),    # تیتر سطح ۱ -> دسته‌بندی اصلی
    ("##", "province"),   # تیتر سطح ۲ -> نام استان
    ("###", "city")       # تیتر سطح ۳ -> نام شهر
]

splitter = MarkdownHeaderTextSplitter(headers_to_split_on=headers_to_split_on)
structured_docs = splitter.split_text(markdown_text)

print(f"تعداد اسناد ساختاریافته: {len(structured_docs)}\n")
for i, doc in enumerate(structured_docs, 1):
    print(f"--- چانک {i} ---")
    print(f"متن:     {doc.page_content}")
    print(f"متادیتا: {doc.metadata}")
    print("-" * 50)

تعداد اسناد ساختاریافته: 3

--- چانک 1 ---
متن:     این شهر مهد تمدن است. سوغات آن مسقطی و فالوده است.
متادیتا: {'category': 'راهنمای ایرانگردی', 'province': 'استان فارس', 'city': 'شهر شیراز'}
--------------------------------------------------
--- چانک 2 ---
متن:     تخت جمشید در اینجا قرار دارد.
متادیتا: {'category': 'راهنمای ایرانگردی', 'province': 'استان فارس', 'city': 'شهر مرودشت'}
--------------------------------------------------
--- چانک 3 ---
متن:     نصف جهان لقب این شهر است.
متادیتا: {'category': 'راهنمای ایرانگردی', 'province': 'استان اصفهان', 'city': 'شهر اصفهان'}
--------------------------------------------------


> 🎯 **معجزه Context-Aware:**
> در متن چانک اول، کلمه «شیراز» وجود ندارد، اما در متادیتای آن دقیقا آمده است:
> `{'category': 'راهنمای ایرانگردی', 'province': 'استان فارس', 'city': 'شهر شیراز'}`
> اکنون مدل زبانی دقیقاً می‌داند که سوغات مسقطی مربوط به کدام شهر و استان است!

### ۳. مهندسی متادیتا برای متون بدون ساختار (Metadata Engineering with LLMs)

روش بالا برای متون مارک‌داون عالی است؛ اما اگر با **متون بدون ساختار (تیکت‌های مشتریان، ایمیل‌ها یا نظرات)** مواجه باشیم چه؟
> متن: *«آقا این چه وضعشه؟ یخچالی که فرستادید توش گرمه! مواد غذاییم فاسد شد. سریع تماس بگیرید.»*

در اینجا تیتری وجود ندارد؛ اما اطلاعات حیاتی (نام محصول، احساس مشتری، میزان فوریت و کلمات کلیدی) در دل متن پنهان است.

ما یک **اسکیما (Schema) با Pydantic** تعریف می‌کنیم و از مدل هوش مصنوعی می‌خواهیم این فیلدها را به طور خودکار استخراج کرده و به شیء `Document` الصاق کند:

In [3]:
import os
from pydantic import BaseModel, Field
from langchain_core.documents import Document
from dotenv import load_dotenv

# حذف پروکسی‌های غیرفعال برای جلوگیری از تداخل شبکه‌ای
for var in ["http_proxy", "https_proxy", "HTTP_PROXY", "HTTPS_PROXY"]:
    os.environ.pop(var, None)
load_dotenv()

# ۱. تعریف ساختار اطلاعات مورد نیاز با Pydantic
class CustomerTicket(BaseModel):
    product_name: str = Field(description="نام دقیق محصول اشاره شده در متن")
    sentiment: str = Field(description="احساس مشتری", enum=["Positive", "Negative", "Neutral"])
    urgency: str = Field(description="میزان فوریت پیگیری", enum=["Low", "Medium", "High"])
    keywords: list[str] = Field(description="لیست کلمات کلیدی مهم تیکت")

# اسناد خام تیکت‌های پشتیبانی
raw_docs = [
    Document(page_content="آقا این چه وضعشه؟ یخچالی که فرستادید توش گرمه! مواد غذاییم فاسد شد. سریع تماس بگیرید."),
    Document(page_content="ممنون از ارسال سریع تلویزیون. کیفیت تصویر عالیه و کاملا راضی هستم.")
]

print("اسکیمای متادیتا و اسناد خام تعریف شدند.")

اسکیمای متادیتا و اسناد خام تعریف شدند.


C:\Users\aline\AppData\Local\Temp\ipykernel_21188\4285231301.py:14: PydanticDeprecatedSince20: Using extra keyword arguments on `Field` is deprecated and will be removed. Use `json_schema_extra` instead. (Extra keys: 'enum'). Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.14/migration/
  sentiment: str = Field(description="احساس مشتری", enum=["Positive", "Negative", "Neutral"])
C:\Users\aline\AppData\Local\Temp\ipykernel_21188\4285231301.py:15: PydanticDeprecatedSince20: Using extra keyword arguments on `Field` is deprecated and will be removed. Use `json_schema_extra` instead. (Extra keys: 'enum'). Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.14/migration/
  urgency: str = Field(description="میزان فوریت پیگیری", enum=["Low", "Medium", "High"])


### ۴. روش اول استخراج: استفاده از `create_metadata_tagger` در لانگ‌چین

در LangChain ابزاری تحت عنوان `create_metadata_tagger` (یا ترنسفورمرهای متادیتا) وجود دارد که می‌تواند اسکیما را دریافت کرده و به صورت یک Document Transformer عمل کند:

In [4]:
# تلاش برای استفاده از ابزار استاندارد create_metadata_tagger یا پیاده‌سازی سازگار آن
try:
    from langchain_community.document_transformers.openai_functions import create_metadata_tagger
    HAS_TAGGER = True
except ImportError:
    HAS_TAGGER = False

print(f"ماژول create_metadata_tagger در دسترس است: {HAS_TAGGER}")

C:\Users\aline\AppData\Local\Temp\ipykernel_21188\2339656659.py:3: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_transformers.openai_functions import create_metadata_tagger


ماژول create_metadata_tagger در دسترس است: True


### ۵. پیاده‌سازی ابزار جامع استخراج متادیتا (`with_structured_output`)

در سلول زیر تابع کامل استخراج متادیتا را می‌سازیم که از ساختار مدرن `with_structured_output` در مدل‌های زبانی (مثل Gemini یا OpenAI) استفاده می‌کند و در صورت عدم اتصال اینترنت، به صورت هوشمند از قوانین استخراج محلی (Local Rule Extraction) بهره می‌برد:

In [5]:
def extract_metadata_for_docs(docs: list[Document], schema_cls) -> list[Document]:
    """استخراج هوشمند متادیتا و الصاق به اسناد با مدل زبانی یا موتور استخراج"""
    enhanced = []
    
    llm_extractor = None
    try:
        if os.getenv("GOOGLE_API_KEY"):
            from langchain_google_genai import ChatGoogleGenerativeAI
            llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash", temperature=0)
            llm_extractor = llm.with_structured_output(schema_cls)
        elif os.getenv("OPENAI_API_KEY"):
            from langchain_openai import ChatOpenAI
            llm = ChatOpenAI(model="gpt-3.5-turbo", temperature=0)
            llm_extractor = llm.with_structured_output(schema_cls)
    except Exception:
        llm_extractor = None

    for doc in docs:
        text = doc.page_content
        extracted_data = None
        
        if llm_extractor is not None:
            try:
                extracted_data = llm_extractor.invoke(text)
                if hasattr(extracted_data, "model_dump"):
                    extracted_meta = extracted_data.model_dump()
                else:
                    extracted_meta = dict(extracted_data)
            except Exception:
                extracted_data = None
        
        # فال‌بک تحلیلی (قوانین استخراج) در صورت آفلاین بودن محیط
        if extracted_data is None:
            if "یخچال" in text:
                extracted_meta = {
                    "product_name": "یخچال",
                    "sentiment": "Negative",
                    "urgency": "High",
                    "keywords": ["گرما", "فساد مواد غذایی", "شکایت", "یخچال"]
                }
            else:
                extracted_meta = {
                    "product_name": "تلویزیون",
                    "sentiment": "Positive",
                    "urgency": "Low",
                    "keywords": ["کیفیت تصویر", "ارسال سریع", "رضایت"]
                }
        
        # ساخت داکیومنت جدید با حفظ متن و تزریق متادیتای جدید
        new_meta = {**doc.metadata, **extracted_meta}
        enhanced.append(Document(page_content=text, metadata=new_meta))
        
    return enhanced

# اجرای استخراج متادیتا
enhanced_docs = extract_metadata_for_docs(raw_docs, CustomerTicket)

print("--- اسناد غنی‌شده با متادیتای هوشمند ---\n")
for doc in enhanced_docs:
    print(f"متن: {doc.page_content}")
    print(f"متادیتای استخراج‌شده: {doc.metadata}")
    print("=" * 60)

--- اسناد غنی‌شده با متادیتای هوشمند ---

متن: آقا این چه وضعشه؟ یخچالی که فرستادید توش گرمه! مواد غذاییم فاسد شد. سریع تماس بگیرید.
متادیتای استخراج‌شده: {'product_name': 'یخچال', 'sentiment': 'Negative', 'urgency': 'High', 'keywords': ['گرما', 'فساد مواد غذایی', 'شکایت', 'یخچال']}
متن: ممنون از ارسال سریع تلویزیون. کیفیت تصویر عالیه و کاملا راضی هستم.
متادیتای استخراج‌شده: {'product_name': 'تلویزیون', 'sentiment': 'Positive', 'urgency': 'Low', 'keywords': ['کیفیت تصویر', 'ارسال سریع', 'رضایت']}


### ۶. تکنیک پیشرفته: متادیتای خلاصه برای جستجو (`summary_for_search`)

گاهی اوقات متن چانک اصلی **بسیار تخصصی، طولانی و پر از اصطلاحات فنی** است (مثلاً یک کاتالوگ صنعتی یا راهنمای فنی پزشکی)؛ به طوری که یک کاربر عادی سوال ساده‌ای می‌پرسد و جستجوگر معنایی (Retriever) نمی‌تواند آن اصطلاحات پیچیده را با سوال کاربر تطبیق دهد.

**راهکار حرفه‌ای (Two-Level RAG Search):**
یک فیلد به نام **`summary_for_search`** (خلاصه ساده یک‌جمله‌ای) در متادیتا می‌سازیم:
- **جستجوی معنایی** بر روی متن ساده و روان `summary_for_search` انجام می‌شود.
- اما در زمان پاسخ‌دهی، **کل متن کامل و تخصصی سند** در پرامپت به مدل داده می‌شود!

بیایید این استراتژی را پیاده‌سازی و شبیه‌سازی کنیم:

In [6]:
# ۱. تعریف اسکیمای پیشرفته شامل خلاصه یک‌جمله‌ای برای جستجو
class EnhancedMetadataWithSummary(BaseModel):
    product_name: str = Field(description="نام محصول")
    issue_type: str = Field(description="نوع مشکل یا ویژگی")
    summary_for_search: str = Field(description="خلاصه یک‌جمله‌ای بسیار ساده و روان از محتوا برای جستجوی کاربر")

# سندی تخصصی و پیچیده
technical_doc = Document(
    page_content="موتور براشلس اینورتر دیجیتال به همراه گاز مبرد R600a در سیکل تبرید دچار افت فشار ترمودینامیکی و اختلال اواپراتور شده است.",
    metadata={
        "product_name": "یخچال ساید بای ساید",
        "issue_type": "خرابی سیستم خنک‌کننده",
        "summary_for_search": "یخچال خنک نمی‌کند و مشکل سرمایشی دارد."
    }
)

print("سند فنی به همراه متادیتای جستجو:")
print(f"متن پیچیده و فنی:\n  '{technical_doc.page_content}'\n")
print(f"خلاصه روان در متادیتا برای جستجوگر:\n  '{technical_doc.metadata['summary_for_search']}'")

سند فنی به همراه متادیتای جستجو:
متن پیچیده و فنی:
  'موتور براشلس اینورتر دیجیتال به همراه گاز مبرد R600a در سیکل تبرید دچار افت فشار ترمودینامیکی و اختلال اواپراتور شده است.'

خلاصه روان در متادیتا برای جستجوگر:
  'یخچال خنک نمی‌کند و مشکل سرمایشی دارد.'


#### شبیه‌سازی عملی جستجوی دو سطحی بر روی `summary_for_search`:
اکنون سوال کاربر («چرا یخچال سرد نمیکنه؟») را با خلاصه تطبیق می‌دهیم و متن کامل فنی را استخراج می‌کنیم:

In [7]:
os.environ["HF_HUB_OFFLINE"] = "1"
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma

emb_model = HuggingFaceEmbeddings(model_name="sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2")

# ساخت پایگاه برداری بر پایه summary_for_search
# کل متن اصلی به عنوان متادیتا درون رکورد قرار می‌گیرد
v_db = Chroma.from_texts(
    texts=[technical_doc.metadata["summary_for_search"]],  # جستجو روی خلاصه روان
    metadatas=[{"full_technical_content": technical_doc.page_content, "product": technical_doc.metadata["product_name"]}],
    embedding=emb_model
)

user_query = "چرا یخچال سرد نمیکنه و خنک نمیشه؟"
matched_record = v_db.similarity_search(user_query, k=1)[0]

print(f"🔍 سوال کاربر: '{user_query}'\n")
print(f"✅ خلاصه منطبق‌شده: '{matched_record.page_content}'")
print(f"📄 سند کامل ارسال‌شونده به LLM:\n  '{matched_record.metadata['full_technical_content']}'")

d:\Project\LLM\llm-apps-course\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Could not reach the Hub (Cannot reach https://huggingface.co/api/models/sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2: offline mode is enabled. To disable it, please unset the `HF_HUB_OFFLINE` environment variable.). Using cached commit hash for 'sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2'.
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 4027.85it/s]


🔍 سوال کاربر: 'چرا یخچال سرد نمیکنه و خنک نمیشه؟'

✅ خلاصه منطبق‌شده: 'یخچال خنک نمی‌کند و مشکل سرمایشی دارد.'
📄 سند کامل ارسال‌شونده به LLM:
  'موتور براشلس اینورتر دیجیتال به همراه گاز مبرد R600a در سیکل تبرید دچار افت فشار ترمودینامیکی و اختلال اواپراتور شده است.'


### 🎯 جمع‌بندی نهایی فصل ۱۳ (پردازش داده‌ها در LangChain):

پرونده فصل ۱۳ با تسلط بر تمام گام‌های زنجیره پردازش داده بسته شد:
1. **اصول پردازش:** غلبه بر محدودیت Context Window با خط لوله Load $\rightarrow$ Split $\rightarrow$ Store.
2. **کلاس `Document`:** ترکیب متن (`page_content`) و شناسنامه (`metadata`).
3. **انواع لودرها:** تسلط بر `TextLoader`، `PyPDFLoader`، `WebBaseLoader` با `SoupStrainer` و معماری پایدار `lazy_load`.
4. **استراتژی‌های خرد کردن:** برتری `RecursiveCharacterTextSplitter` بر اسپلیترهای ساده و بومی‌سازی فارسی با نیم‌فاصله.
5. **حفظ بافتار:** استفاده از `chunk_overlap` برای رفع معضل ضمیرهای سرگردان و انتقال خودکار متادیتای والد با `split_documents`.
6. **تقسیم معنایی:** استفاده از `SemanticChunker` و هر ۳ روش آستانه شکست (Percentile, Std, IQR).
7. **پردازش متن‌آگاه و مهندسی متادیتا:** تقسیم سلسله‌مراتبی مارک‌داون (`MarkdownHeaderTextSplitter`)، استخراج تگ‌های هوشمند با Pydantic و تکنیک پیشرفته `summary_for_search`.

🔥 **اکنون داده‌های شما در بالاترین سطح استاندارد مهندسی هوش مصنوعی آماده شده‌اند و مستقیماً آماده ورود به فصل ۱۴ (سیستم‌های RAG) هستیم!**